This is part 14 of a tutorial series. We recommend following them in order, starting with [Part 0: Welcome to `musica`](0.%20Welcome%20to%20MUSICA.ipynb).

# Introduction to Aerosol and Cloud chemistry modeling with `musica`

This tutorial chapter introduces the basics of aerosol/cloud chemistry modeling with `musica`. We will cover:
- "phases" as a conceptual building block of multi-phase chemistry modeling
- how reactions and condensation/evaporation processes are associated with phases
- how to define an aerosol representation that implements phases
- how phase instances are used to build the ODE solver state

Subsequent tutorials will build on these concepts to cover more advanced topics including Differential Algebraic Equation (DAE) solvers for handling steady-state approximations, full cloud chemistry mechanisms, and performance optimizations.

## 1. Importing Libraries

First, let's import the required libraries for this tutorial

In [ ]:
import musica
import musica.mechanism_configuration as mc
from musica.micm import MICM, SolverType, SolverState
import matplotlib.pyplot as plt
import numpy as np

## 2. Species and phases

In `musica` species are combined into "phases" before they are used to define a chemical system. We've seen this already with gas-phase-only systems, which always include exactly one phase named "gas". 

Before we explore condensed phases, let's start with the same simple gas-phase configuration we've used in previous tutorial chapters

In [ ]:
A = mc.Species(name="A", molecular_weight_kg_mol=0.024, density_kg_m3=987.0)
B = mc.Species(name="B", molecular_weight_kg_mol=0.043, density_kg_m3=1001.3)
C = mc.Species(name="C")
gas_species = [A, B, C]
gas = mc.Phase(name="gas", species=[
    mc.PhaseSpecies(A, diffusion_coefficient_m2_s=1.3e-9),
    mc.PhaseSpecies(B, diffusion_coefficient_m2_s=2.3e-9),
    C,
])

This looks a little different than previously. Instead of just adding the species directly to the `gas` phase, we turn some of them into `PhaseSpecies`. This allows us to set properties for a species that are specific to a phase. In this case, the diffusion coefficient is set for `A` and `B`, and only applies to them in the gas-phase.

Let's follow the exact same approach to define a couple condensed phases

In [ ]:
D = mc.Species(name="D", molecular_weight_kg_mol=0.023, density_kg_m3=993.0)
E = mc.Species(name="E", molecular_weight_kg_mol=0.052, density_kg_m3=1002.1)
F = mc.Species(name="F", molecular_weight_kg_mol=0.019, density_kg_m3=996.4)
solvent = mc.Species(name="solvent", molecular_weight_kg_mol=0.018, density_kg_m3=1000.0)
foo_species = [A, E, F, solvent]
bar_species = [B, D, E, F, solvent]
foo = mc.Phase(name="foo", species=foo_species)
bar = mc.Phase(name="bar", species=bar_species)

Any species that is included in a condensed phase (`foo` and `bar` in our toy system) need to have a molecular weight and density defined.

Notice that we can include one species in as many phases as we wish, including the gas phase. For species that partition between the gas and condensed phases, this can be convenient becuase they share the same intrinsic physical properties (e.g., molecular weight), which only need to be defined once.

Whether you use a single species like `SO2` in gas and condensed phases or whether you have species like `SO2_g` and `SO2_aq` that only appear in one phase each is entirely up to you.

We've included a species named `solvent` in the condensed phases. This is because the condensed-phase reactions we'll include are for solutions and need a solvent to be specified. In real atmosphere systems, the solvent is always water. Future development will introduce reactions that occur in a mixture based on mole fraction, to permit including organic aerosol chemistry.

## 3. Chemical Reactions and phases

Next, let's look at how reactions are associated with phases. This is also something we've seen in previous tutorial chapters, but did not discuss in detail. Let's start with our go-to gas-phase reactions

In [ ]:
gas_r1 = mc.Arrhenius(
    name="A_to_B",
    A=1.0e-3,
    C=50,
    reactants=[A],
    products=[B],
    gas_phase=gas,
)

gas_r2 = mc.Arrhenius(
    name="B_to_C",
    A=1.0e-3,
    C=50,
    reactants=[B],
    products=[C],
    gas_phase=gas,
)

Notice that each of our reactions is excplicitly associated with the gas-phase. Next let's create some condensed-phase reactions

In [ ]:
foo_r1 = mc.DissolvedReaction(
    rate_constant=mc.Arrhenius(A=1.3e2, C=-12.3),
    reactants=[A, F],
    products=[E],
    solvent=solvent,
    phase=foo,
)
bar_r1 = mc.DissolvedReaction(
    rate_constant=mc.Arrhenius(A=3.7e2, C=-5.4),
    reactants=[B, D],
    products=[E],
    solvent=solvent,
    phase=bar,
)
bar_r2 = mc.DissolvedReversibleReaction(
    forward_rate_constant=mc.Arrhenius(A=22.3, C=-13.4),
    equilibrium_constant=mc.Equilibrium(A=0.2, C=-1.2, T0=295.0),
    reactants=[E],
    products=[F, solvent],
    solvent=solvent,
    phase=bar,
)

These are two reaction types we haven't seen before:
* `DissolvedReaction`:  a chemical reaction in a condensed phase that takes a rate constant type and who's rate is calculated as the rate constant (calculated for the current environmental conditions) times the product of the reactant mole fraction in solution `[mol m-3(solute) mol-1 m3(solvent)]`.
* `DissolvedReversibleReaction`: a reversible reaction (equivalent to two `DissolvedReaction`s with swapped reactants and products). For a reversible reaction, you must specify exactly two of: forward rate constant, reverse rate constant, and equilibrium rate constant.

Notice that each reaction definition includes the phase it will be associated with. Also notice that the solvent species is treated as any other species and is available to participate in reactions.

## 6. Condensation/Evaporation

We've added gas-phase reactions and condensed-phase reactions, but for this to truly be a multi-phase system we need a way for species to partition between the gas and condensed phases.

`musica` currently has one option for condensation/evaporation of gases: Henry's Law partitioning. However, a vapor-pressure based partitioning process useful for organic aerosols is in-the-works.

Let's allow species `A` and `B` to partition between the gas phase and the phases `foo` and `bar` respectively

In [ ]:
M_ATM_TO_MOL_M3_PA = 1000.0 / 101325.0 #  convert from M atm-1 to base SI units

a_transfer = mc.HenrysLawPhaseTransfer(
    gas_phase=gas,
    gas_species=A,
    condensed_phase=foo,
    condensed_species=A,
    solvent=solvent,
    henrys_law_constant=mc.HenrysLawConstant(
        HLC_ref=123.2 * M_ATM_TO_MOL_M3_PA,
        C=-2310.0,
    ),
    accommodation_coefficient=0.5,
)
b_transfer = mc.HenrysLawPhaseTransfer(
    gas_phase=gas,
    gas_species=B,
    condensed_phase=bar,
    condensed_species=B,
    solvent=solvent,
    henrys_law_constant=mc.HenrysLawConstant(
        HLC_ref=329.1 * M_ATM_TO_MOL_M3_PA,
        C=-3124.0,
    ),
    accommodation_coefficient=1.0,
)


## 5. Aerosol Representations

Aerosol microphysical models typically assume a specific algorithm for representing aerosol populations (e.g., log-normal modes, sections, single-particles). Here's a cartoon representation from an under-cited paper about mulit-phase chemistry modeling

<p align="center">
  <img src="assets/aero_reps.png" alt="aerosol representationst">
</p>

`musica` is designed to work with a variety of such representations, making it compatible with existing aerosol microphysical models. Solving condensed-phase chemistry and condensation/evaporation of gases only requires a few specific pieces of information about the aerosol state at a given point in time (e.g., surface area density, number concentration). `musica` employs adapters for aerosol representations that we can include to fully define our multi-phase system and make this information available to the solver.

You can use as many types and instances of aerosol representations as you like. For example, you could configure `musica` to use a 6-mode, 20-section scheme where the modes handle sulfate and organic aerosol and two overlapping 10-bin section sets handle dust and sea-salt. We'll keep things simple for this example and just use a 2-two-moment-modal scheme

In [ ]:
little_mode = mc.TwoMomentMode(
    name="the little mode",
    phases=[foo],
    geometric_standard_deviation=1.2,
)
big_mode = mc.TwoMomentMode(
    name="the big mode",
    phases=[bar, foo],
    geometric_standard_deviation=1.4,
)

Note that we've included two phases in the second mode, and we've included the `foo` phase in both modes. It is up to the aerosol representation how to implement the phases to form particles. The `TwoMomentMode` (as well as the `SingleMomentMode` and `UniformSection`) implement each phase once and arrange them as a sort-of 3-dimensional pie chart, where the relative volumes of each phase equals its relative exposed surface area. This is the configuration used in many aerosol microphysical models.

However, adapters could be added to `musica` for more complex cases. For example, an onion model (with internal layers to approximate the effects of condensed-phase diffusion) could implement each phase once per layer leaving only the outer layer phases exposed to the gas-phase and avaialble for condensation/evaporation. This would only require the introduction of a new aerosol representation adapter in `musica`.

## 6. Create a Multi-phase Chemistry Solver

Let's create a solver for our multi-phase system

In [ ]:
mechanism = mc.Mechanism(
    name="toy multi-phase system",
    species=[A, B, C, D, E, F, solvent],
    phases=[gas, foo, bar],
    reactions=[gas_r1, gas_r2],
    aerosol=mc.Aerosol(
        representations=[little_mode, big_mode],
        processes=[a_transfer, b_transfer, foo_r1, bar_r1, bar_r2],
    ),
)
solver = MICM(
    mechanism=mechanism,
    solver_type=SolverType.rosenbrock,
    external_models=[musica.MIAM()]
)

The `mechanism` brings everything together: the full set of species, the full set of phases, the gas-phase reactions, the aerosol representation, the condensed-phase reactions, and the condensation/evaporation processes.

Now that the solver state has been created, let's take a look at how it's structured

In [ ]:
state = solver.create_state()
ordering = state.get_species_ordering()
# sort by name to see variables organized by representation and phase
for name, idx in sorted(ordering.items(), key=lambda x: x[0]):
    print(f"[{idx:2d}] {name}")


This shows how `musica` assembles the multi-phase system we've defined into a set of solver variables. Because we're using two-moment modes, you'll notice that each mode has a number concentration associated with it. Even though none of our processes affect number concentration, new ones might. This allows representations with dynamically variable number concentrations to be comaptible with `musica`. Single-moment representations have number concentrations calculated based on total mass and would not have this solver variable present.

Also notice the two implementations of the `foo` phase - one in each mode. Processes affecting the `foo` phase will be applied to both instances. For the partitioning of `A` to `foo` from the gas phase, both `foo` instances compete for uptake of the single `foo` gas-phase species.

## 7. Set initial conditions

Now that we have a solver and state, let's set some initial conditions

In [ ]:
state.set_concentrations({
    "A": 1.2,
    "B": 2.3,
    "the big mode.NUMBER_CONCENTRATION": 1.0e5,
    "the big mode.bar.D": 2.0e-3,
    "the big mode.bar.solvent": 7.0e-3,
    "the big mode.foo.F": 3.0e-3,
    "the big mode.foo.solvent": 4.0e-2,
    "the little mode.NUMBER_CONCENTRATION": 1.0e6,
    "the little mode.foo.F": 6.0e-4,
    "the little mode.foo.solvent": 1.0e-3,
})
state.set_conditions(
    temperatures=265.0,
    pressures=101321.3,
)
mechanism.aerosol.set_default_parameters(state)

The last line is something we haven't seen before. When including aerosols in a solve, we need to call the `set_default_parameters` function to set the aerosol representation parameters we defined above. In our case, since we're using two-moment modes, this sets the geometric standard deviation (GSD) of the two modes.

## 8. Solve and Plot

Let's run the solver over a few time steps and plot the results

In [ ]:
DT = 0.1 # one second time steps
TARGET_TIME = 3600.0 # 60 min simulation

times = [0.0]
results = [state.get_concentrations()]

total_time = 0.0
while total_time < TARGET_TIME - 1.0e-10:
    result = solver.solve(state, time_step=DT)
    assert result.state == SolverState.Converged, \
        f"Solver failed at t={total_time:.4f}s (state={result.state})"
    total_time += DT

    times.append(total_time)
    results.append(state.get_concentrations())

Let's first take a look at the gas-phase state. We should see a similar `A -> B -> C` type pattern, with competitive partitioning and condensed-phase reaction of `A` and `B`

In [ ]:
# time series extraction helper
def ts(name):
    return np.array([r[name][0] for r in results[1:]])

t = np.array(times[1:])

fig, ax = plt.subplots()
ax.plot(t / 60, ts("A"), label="A")
ax.plot(t / 60, ts("B"), label="B")
ax.plot(t / 60, ts("C"), label="C")
ax.set_xlabel("Time (min)")
ax.set_ylabel("Concentration (mol m⁻³)")
ax.set_title("Gas-phase species")
ax.legend()
plt.tight_layout()
plt.show()


Now, let's look at the evolution of the condensed-phase species

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

panels = [
    {
        "title": "little mode — foo phase",
        "species": ["the little mode.foo.A", "the little mode.foo.E", "the little mode.foo.F"],
        "labels": ["A", "E", "F"],
    },
    {
        "title": "big mode — foo phase",
        "species": ["the big mode.foo.A", "the big mode.foo.E", "the big mode.foo.F"],
        "labels": ["A", "E", "F"],
    },
    {
        "title": "big mode — bar phase",
        "species": ["the big mode.bar.B", "the big mode.bar.D", "the big mode.bar.E", "the big mode.bar.F"],
        "labels": ["B", "D", "E", "F"],
    },
]

for ax, panel in zip(axes, panels):
    for key, label in zip(panel["species"], panel["labels"]):
        ax.plot(t / 60, ts(key), label=label)
    ax.set_xlabel("Time (min)")
    ax.set_ylabel("Concentration (mol m⁻³)")
    ax.set_title(panel["title"])
    ax.legend()

plt.tight_layout()
plt.show()
